# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra-Jahangir/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


**Lane 3: Structured Content Archetype Clustering**

This is a **clustering** task because there is no known target label that says what type of page each content item belongs to. Instead, the goal is to group content items that have similar observable performance and content characteristics.

For this lane, one row represents a content item, and the model will use structured signals such as visibility, traffic, position, engagement, content age, and content size to discover recurring performance archetypes.

The output is therefore not a prediction of a pre-existing label. It is a set of clusters that can be profiled and translated into useful content actions.

In [1]:
# Load the starter playground and inspect the available fields.
# Run this notebook from the repository so the relative path resolves correctly.

from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Starter dataset not found at {DATA_PATH}. "
        "Run this notebook from work/notebooks/ inside the FlyRank starter repository."
    )

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Available columns:")
print(df.columns.tolist())


Rows: 30000
Columns: 44
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Clustering does not require a supervised target column. The **proxy/output** for this task is the **cluster membership assigned to each content item**.

The clusters are intended to represent recurring performance archetypes, such as strong-performing pages, high-visibility but weak-engagement pages, stale pages, or low-demand pages. I will not assign those names before seeing the data; the cluster profiles should determine how each group is described.

The cluster assignment is therefore a discovered structure rather than a ground-truth label. The action supported by the output is to map each observed archetype to a review action such as **protect, improve, rewrite, merge, prune, or monitor**.



In [2]:
# Define structured, observable features for clustering.
# We avoid IDs and any product decision fields. We also avoid raw text/URLs.

candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "content_age_days",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

feature_cols = [c for c in candidate_features if c in df.columns]

if len(feature_cols) < 4:
    raise ValueError(
        "Not enough expected structured features were found. "
        f"Found: {feature_cols}"
    )

cluster_df = df[feature_cols].copy()

# Keep rows with at least some evidence of visibility, matching the starter
# playground's basic idea of avoiding completely unobserved pages.
if "impressions_90d" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["impressions_90d"].fillna(0) > 0].copy()

# Show the actual unit of analysis.
display(cluster_df.head(10))
print("Unit of analysis: one row = one content item.")
print("Rows used for clustering:", len(cluster_df))
print("Features used:", feature_cols)


,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,content_age_days,word_count,char_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
0,3803,29,17,0,187,3221.0,20457.0,0.76,10.6,5.88,4.55,0.0
1,15320,7,9,0,445,2481.0,15562.0,0.05,20.3,0.00,10.00,0.0
2,12581,11,11,0,141,3515.0,23643.0,0.09,36.5,0.00,28.57,0.0
3,11751,58,78,0,463,NaN,NaN,0.49,6.2,1.28,3.45,0.0
4,19140,24,145,0,263,2803.0,17469.0,0.13,44.0,0.00,24.29,0.0
5,3970,1,5,0,147,3080.0,18178.0,0.03,8.5,0.00,25.00,0.0
6,20,0,1,0,90,3059.0,20810.0,0.00,7.0,0.00,0.00,0.0
7,1724,1,28,0,445,NaN,NaN,0.06,21.2,3.57,7.14,0.0
8,32574,29,68,0,90,3807.0,24228.0,0.09,46.0,5.88,6.25,0.0
9,1240,2,3,0,257,NaN,NaN,0.16,4.9,0.00,0.00,0.0


Unit of analysis: one row = one content item.
Rows used for clustering: 30000
Features used: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'content_age_days', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 3. Success metric

The primary model-quality metric is the **Silhouette Score**.

Silhouette Score measures whether each content item is more similar to the other items in its own cluster than to items in neighboring clusters. A higher score indicates that the clusters are more clearly separated and internally coherent.

For this exploratory clustering task, I will compare several values of *k* and use the silhouette score as a diagnostic rather than treating a single score as proof that the archetypes are "true." The final choice should also be interpretable and useful for content review.

**Decision-support success:** the clusters should be reasonably separated, stable enough to describe, and distinct enough to support different content actions.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Log-transform highly skewed count/volume signals before scaling.
X = cluster_df.copy()

log_cols = [
    c for c in [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ai_sessions_90d",
        "content_age_days",
        "word_count",
        "char_count",
    ]
    if c in X.columns
]

for c in log_cols:
    X[c] = np.log1p(X[c].clip(lower=0))

preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_scaled = preprocess.fit_transform(X)

results = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    results.append({"k": k, "silhouette_score": score})

scores = pd.DataFrame(results)
display(scores)

best_k = int(scores.loc[scores["silhouette_score"].idxmax(), "k"])
print("Best silhouette score:", round(scores["silhouette_score"].max(), 3))
print("Candidate k selected for inspection:", best_k)



,k,silhouette_score
0,2,0.198556
1,3,0.209044
2,4,0.200833
3,5,0.196780
4,6,0.208301
5,7,0.216227
6,8,0.212400


Best silhouette score: 0.216
Candidate k selected for inspection: 7


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content item/page**.

This is important because the lane asks which performance archetypes exist **across the content inventory**. The clustering should therefore compare content items to one another, rather than treating each daily performance record as a separate page.

The starter data contains observable signals and pseudonymized identifiers. For clustering, the identifier is only useful for tracing the assigned cluster back to a content item; it is not used as a numeric feature.


In [4]:
# Fit the selected clustering model and attach the discovered cluster to each content item.

final_model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_labels = final_model.fit_predict(X_scaled)

clustered = cluster_df.copy()
clustered["cluster"] = cluster_labels

# Add a safe pseudonymized content identifier for inspection if available.
if "content_id" in df.columns:
    clustered.insert(
        0,
        "content_id",
        df.loc[cluster_df.index, "content_id"].values
    )

display(clustered.head(15))
print("One row = one content item; 'cluster' is the discovered archetype membership.")
print("\nCluster sizes:")
display(clustered["cluster"].value_counts().sort_index().rename("content_items").to_frame())


,content_id,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,content_age_days,word_count,char_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,cluster
0,content_304f48230142,3803,29,17,0,187,3221.0,20457.0,0.76,10.6,5.88,4.55,0.0,1
1,content_a1fb4e703a9e,15320,7,9,0,445,2481.0,15562.0,0.05,20.3,0.00,10.00,0.0,0
2,content_9aa793d4d895,12581,11,11,0,141,3515.0,23643.0,0.09,36.5,0.00,28.57,0.0,1
3,content_331d6c4de07b,11751,58,78,0,463,NaN,NaN,0.49,6.2,1.28,3.45,0.0,1
4,content_d99b7a2d90ca,19140,24,145,0,263,2803.0,17469.0,0.13,44.0,0.00,24.29,0.0,1
5,content_d4084a4bc775,3970,1,5,0,147,3080.0,18178.0,0.03,8.5,0.00,25.00,0.0,4
6,content_9a34b442b552,20,0,1,0,90,3059.0,20810.0,0.00,7.0,0.00,0.00,0.0,4
7,content_a63219c6e95a,1724,1,28,0,445,NaN,NaN,0.06,21.2,3.57,7.14,0.0,0
8,content_5e6c160719bc,32574,29,68,0,90,3807.0,24228.0,0.09,46.0,5.88,6.25,0.0,1
9,content_c27558df2b0c,1240,2,3,0,257,NaN,NaN,0.16,4.9,0.00,0.00,0.0,0


One row = one content item; 'cluster' is the discovered archetype membership.

Cluster sizes:


,content_items
cluster,
0,7415
1,8006
2,158
3,4234
4,8771
5,144
6,1272


In [5]:
# Profile the clusters using the original (untransformed) values.
profile_cols = [c for c in feature_cols if c in clustered.columns]

cluster_profile = (
    clustered.groupby("cluster")[profile_cols]
    .median(numeric_only=True)
    .round(2)
)

display(cluster_profile)

print(
    "Interpretation note: cluster names should be assigned only after inspecting "
    "these profiles. The numbers above are descriptive, not causal."
)


,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,content_age_days,word_count,char_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
cluster,,,,,,,,,,,,
0,561.0,0.0,5.0,0.0,438.0,2883.0,18364.5,0.00,20.40,0.00,0.00,0.00
1,6600.0,16.0,44.0,0.0,228.0,3060.0,20179.0,0.26,8.10,1.85,5.48,0.00
2,3.0,1.0,1.0,0.0,296.5,894.0,6397.0,33.33,3.50,0.00,0.00,0.00
3,28.0,0.0,2.5,0.0,294.0,1392.0,9007.5,0.00,8.30,0.00,14.29,0.00
4,139.0,0.0,3.0,0.0,131.0,3033.0,20649.0,0.00,9.10,0.00,17.19,0.00
5,115.0,0.0,2.0,2.0,243.5,2892.0,19182.0,0.00,8.85,0.00,9.43,66.67
6,11621.5,30.0,145.0,2.0,229.0,5561.5,36299.0,0.26,18.95,2.10,4.95,2.13


Interpretation note: cluster names should be assigned only after inspecting these profiles. The numbers above are descriptive, not causal.


## 5. Why ML beats a fixed rule here

A fixed rule can identify one hand-defined pattern, such as "high impressions and low CTR," but Lane 3 is asking a broader question: **what recurring performance archetypes exist across the content inventory?**

The content inventory contains several interacting signals — visibility, clicks, sessions, position, engagement, age, and content size. It is difficult to write a small set of mutually exclusive `if/elif` rules that discovers all meaningful combinations without deciding the archetypes in advance.

Clustering is useful here because it can group pages from their observed feature patterns without requiring us to define the categories beforehand. This is an **analysis and decision-support tool**, not a claim that the clusters are natural or permanent types.

The output supports content work by giving reviewers a smaller set of page archetypes to inspect and by suggesting different actions for different profiles. The action mapping should remain evidence-based and should be reviewed by a human.

In [6]:
# Simple action-mapping scaffold.
# The exact archetype names are intentionally not hard-coded before inspecting
# the cluster profiles.

action_map = pd.DataFrame({
    "cluster": sorted(clustered["cluster"].unique()),
    "observed_archetype": ["Inspect profile first"] * clustered["cluster"].nunique(),
    "possible_action": ["protect / improve / rewrite / merge / prune / monitor"] * clustered["cluster"].nunique(),
})

display(action_map)

print(
    "Next step for a full lane analysis: inspect each cluster profile, give it a "
    "plain-language archetype name, and map that profile to a justified action."
)


,cluster,observed_archetype,possible_action
0,0,Inspect profile first,protect / improve / rewrite / merge / prune / ...
1,1,Inspect profile first,protect / improve / rewrite / merge / prune / ...
2,2,Inspect profile first,protect / improve / rewrite / merge / prune / ...
3,3,Inspect profile first,protect / improve / rewrite / merge / prune / ...
4,4,Inspect profile first,protect / improve / rewrite / merge / prune / ...
5,5,Inspect profile first,protect / improve / rewrite / merge / prune / ...
6,6,Inspect profile first,protect / improve / rewrite / merge / prune / ...


Next step for a full lane analysis: inspect each cluster profile, give it a plain-language archetype name, and map that profile to a justified action.


## Self-check

- [x] The ML task type is named: **clustering**.
- [x] The output/proxy is defined: **cluster membership / content archetype**.
- [x] A success metric is named: **Silhouette Score**, supported by interpretability.
- [x] The unit of analysis is explicit: **one row = one content item**.
- [x] The notebook loads the lane's starter data and shows a real dataframe.
- [x] The approach uses observable structured signals rather than product decisions.
- [x] The reason ML is useful is explained without claiming that clusters are ground-truth labels.
- [x] The output is tied to a content action: archetype-based review and prioritization.
- [x] No client names, URLs, private queries, raw titles, or other private-origin fields are intentionally displayed.
- [ ] Run **Runtime → Run all** from the repository so the starter CSV is available at the expected relative path.
- [ ] Commit the executed notebook to `work/notebooks/w02_ml_task_framing.ipynb`.
